<a href="https://colab.research.google.com/github/snr-laboratory/snrlab-ic-q-pix-v1/blob/chip_network_sim/chip_network_sim/dev_journals/kgosine_202603_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Purpose


Simulate an m x n digital chip network where each chip is described with RTL. Each chip rceives data from one upstream neighbor and sends data to one downstream neighbor. Each chip also generates data locally. Data traffic is buffered using a single local FIFO (defined via RTL). The routing of data through the network is defined via the Orchestrator.

The system is a distributed synchronous hardware simualtor where there is a global clock, chips which act as hardware modules, and NNG sockets which act as wires between hardware modules.

## Chip Unit

A chip functions as a 2-input, 1-ouput unit. The first input is data from one upstream chip. The second input is data generated locally on the chip. The one output is passed to the downstream chip.

The FIFO on the chip is defined with RTL (2-input, 1-output) which buffers data traffice through the chip.

The locally generated data can be viewed as any analog or mixed-signal front-end structure within the chip which generates data-packets. In this example, this front-end is generalized to a random number generator

Generated packets:
lines [666:680] chip_rtl.cpp
A packet is generated when a RNG selects a draw < gen_ppm where gen_ppm is an int between 0-1,000,000 that is passed as an argument to each chip. gen_ppm acts as a probability (rate) at which packets are generated.

The contents of a data packet are the following 64-bits:
src_id: 16 bits for chip id
timestamp: 24 bits for timestamp
payload: 24 bits for data payload

The payload is a randomly generated 24 bits. The timestamp is incrememented with each clock pulse.

## Config

run_from_config.py is an orchestrator launcher which passes arguments defined in a config.json to the Orchestrator (orchestrator.c). See network_3x4_snake_to_top_left.json for an example of how the routing, data packet, fifo_depth, and data packet generation can be set.


## Orchestrator

orchestrator.c is the top-level simualtion driver. it parses run settings (which can be passed using config.json) and then launches one chip process per grid cell. It is responsible for running the lock-step control protocol; it passes a TICK to all chips and waits to collect DONE from all chips, enforcing that all chips are stepping together in time. The orchestrator will use nng sockets for orchestrator-to-chp control/metrics to confirm time-stepping.



## Message Transport

The program uses NNG (nanomsg-next-gen) as a lightweight broker-less API to handle the movement of data packets between chips. There are three NNG channels defined in the program:

* REP socket: clock channel

Used for synchronization with the central clock driver provided by the orchestrator; sends each chip a TICK message which cues the chip simulation to run one cycle and reply with DONE message.

[ Clock (REQ) --TICK--> Chip (REP) ]

Each chip executes one simulation step

[ Chip (REP) --DONE--> Clock (REQ) ]

with added enforcement to WAIT until all chips reply DONE to send next tick, ensuring synchronization.


* REQ/REP socket: data channel between chips

Used for exchange of data packets between neighboring chips. The behavior depends if the chip is outputting or inputting data:

(a) If a chip is passing data downstream, REP server is set up. Waits for a DATA_PULL request from the downstream chip, then will respond with DATA_REPLY. In short, the downstream chip pulls data for at a specific TICK and the server answers when the data at that TICK is ready. If the chip did not produce an output packet at that tick, it will send nothing.

[ Downstream Chip (REQ) --DATA_PULL-->  Chip (REP) ]

[ Chip (REP) --DATA_REPLY--> Downstream Chip (REQ) ]

(b) If the chip is reading from upstream is opens a REQ socket. Each tick it will request a data packet from upstream, wait for a reply from the upstream chip, then validates that the data packet which arrived has the correct upstream chip id.

[ Chip (REQ) --DATA_REQUEST--> Upstream Chip (REP) ]

[ Upstream Chip (REP) --DATA_REPLY--> Chip (REQ) ]

* PUSH socket: metrics channel which sends one-way reporting to a metrics colelctor

No blocking or wait for reponse. Used to gather metrics about the simulation.


On each TICK:
* The orchestrator sends TICK(seq) over the REP socket (chips recieve)
* A downstream chip dends DATA_PULL request (REQ socket). The upstream chip's data_server_tread_main() receives the pull requeston its REP socket. It then sends a data packet (should one already be published by the main simulator) over its REP socket to the REQ socket of the downstream chip.

Note that requests made all have the specifi sequence (tick) number. This ensures that chips which are simulating at different speeds do not leak duture information backwards in time. eg if Chip A is simualting faster than Chip B, Chip B could request the data packet for its current tick number (10). If it were to request simply the latest packet from Chip A, which is already on tick 12, it could receive a packet from cycle 12 when it should have received for cycle 10. Therefore, we ensure that requests specify the cycle number, accounting for differences in chip simualtion times across CPUs.

## Future Directions


* incorporating a Spice (analog) portion into the simulation

we've previously successfully performed Spice/RTL cosimulation using Verilator before

* allowing data input from any of the four adjacent chips

this will allow for configuration routes beyond daisy chaining which is obviously vulnerable to single point failure. at this stage, the routing would still be defined at the top level.

* incorpoarting existing RTL (LArPix) including more realistic data packet creation and transmission timing.



## Applications

* because of the distributed network design of the architecture, chips on the order of those needed in DUNE: for 4x4mm pixels, with 62,500 pixels per m^2. Assuming 64 channel chips, on the order of 10^3 chips/ m^2. Assuming pixelating just one face of a DUNE module, that's 380 m^2 so somewhere around 380,000 chips. the only way to do a simualtion of this scale is with distributed networking.

* taking a design from RTL/Spice (transistor-level) to a tile-level system reduces the number of edge cases which are missed. No part of the simualtion path from chip to tile is simplified, increasing confidence of final product. any unintended effects from the transistor-level or communication design wil appear in the simulation results.

* modifiable to any generic chip: highlighted with this toy model (and later by incorporating analog components), this architecture allows for plugging in any generic chip design or chip routing scheme. Further, the communication system can be tweaked for ACK/NACK signals to be included allowing for testing of new networking schemes with minimal changes to the software. The chip design needs only to be compiled with Verilator once and then can quickly be used in a tile simualtion.

* simple simualtions can show packet loss as a function of FIFO sizes, maximum data generation rates, routing schemes ect.



## Logic Elements

(1) Backpressure via the global clock barrier

Because a chip cannot send DONE until it has finished its simulation tick, anything that gets held up during the TICK pushes back on the whole system. One slow chip slows the entire simulation.

(2) Backpressure on the pull-based data transfer

The downstream chip clint requests data for a specific tick. The simulation is stalled until the upstream chip as simulated and published data for that sequence. If the upstream chip is running behind, the server thread is blocked.

(3) Dropping implemented in the FIFO. If the FIFO is full, incoming packets are dropped.

(4) Backpressure on pass-through packet generation. Locally generated packets take priority for FIFO entry. Therefore, pass through packets will 'hold' and wait to enter the FIFO after the local packet has entered (reducing availability by 1). If both are trying to get into the FIFO at the same time and there is 2 or more available positions in the FIFO, both will enter on the SAME tick. Therefore, arrival of both local and pass through packets at the same time will not cause a data loss unless the FIFO has 0 or 1 position available.

## Test Cases


(1) The transport is pull-based but storage is single-slot. Correctness relies on seq matching + lockstep. If lockstep breaks, there is no buffering to recover old packets. eg if Chip B (slow) is still working on tick N (hasn't sent DONE) and chip A has published for tick N but then B requests for a tick that doesn't match what Chip A has retained. This would indicate a failure in the lockstep mechanism.

Possible test: artificially slow the downstream so it requests "old" seq. If the clocking barrier is strict this should just slow down the global simualtion.



## Simulation Metrics


cycles/sec run time ect